# One-Click Colab Runner

Use this when the repo folder is already in Google Drive. After selecting a GPU runtime, `Runtime -> Run all` will install dependencies, prepare Flickr8k, verify the paper soft-attention hyperparameters, train, run validation and test evaluation, and generate attention visualizations.

Manual setup before running:
1. Set Colab runtime to GPU.
2. Put this repo folder in `MyDrive/Neural_Image_Caption_Generator`.
3. Put Flickr8k in `MyDrive/Neural_Image_Caption_Generator/data/flickr8k`, or set `FLICKR8K_SOURCE_DIR` below to another Drive folder containing the images and captions.

The run uses the Flickr8k soft-attention settings from the paper path implemented in this repo: VGG-19/OxfordNet annotations, 512-dimensional embeddings/LSTM/attention, vocabulary size 10,000 total tokens, batch size 64, RMSProp, dropout 0.5, doubly stochastic regularization weight 1.0, no encoder fine-tuning, and the standard 6000/1000/1000 train/validation/test split.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import os

# Change this only if your repo folder has a different name/location.
REPO_DIR = Path('/content/drive/MyDrive/Neural_Image_Caption_Generator')

# If Flickr8k is already inside the repo at data/flickr8k, leave this blank.
# If it is somewhere else in Drive, set it, e.g. Path('/content/drive/MyDrive/flickr8k').
FLICKR8K_SOURCE_DIR = None

# Rebuild vocab.json when an older notebook run created a non-paper 10004-token vocab.
REBUILD_STALE_VOCAB = True

assert REPO_DIR.exists(), f'Repo folder not found: {REPO_DIR}'
os.chdir(REPO_DIR)
print('Working in', Path.cwd())


In [ ]:
!python -m pip install -q -r requirements.txt
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))


## Verify Paper Hyperparameters

This fails fast if `config.py` drifts away from the paper-faithful Flickr8k soft-attention run used by the notebook.


In [ ]:
import config

expected = {
    'EMBED_DIM': 512,
    'DECODER_DIM': 512,
    'ATTENTION_DIM': 512,
    'ENCODER_DIM': 512,
    'DROPOUT': 0.5,
    'VOCAB_SIZE': 10_000,
    'LAMBDA': 1.0,
    'GRAD_CLIP': 5.0,
    'LEARNING_RATE': 4e-4,
    'BATCH_SIZE': 64,
    'NUM_EPOCHS': 50,
    'MAX_DECODE_LEN': 50,
}
actual = {name: getattr(config, name) for name in expected}
assert actual == expected, {'expected': expected, 'actual': actual}
assert config.ENCODER_FINETUNE_EPOCH > config.NUM_EPOCHS, 'paper run should keep the encoder frozen'
print('Paper hyperparameters verified:')
for name, value in expected.items():
    print(f'  {name} = {value}')
print(f'  ENCODER_FINETUNE_EPOCH = {config.ENCODER_FINETUNE_EPOCH} (disabled for 50 epochs)')


### Optional Kaggle Download Path

Only use these two cells if Flickr8k is not already in Drive. In Kaggle, create an API token and upload `kaggle.json` here before running the download cell.


In [ ]:
# OPTIONAL: upload kaggle.json, then run the next cell.
# from google.colab import files
# files.upload()
# !mkdir -p ~/.kaggle
# !cp kaggle.json ~/.kaggle/kaggle.json
# !chmod 600 ~/.kaggle/kaggle.json

In [ ]:
# OPTIONAL: Kaggle download. Uncomment only if you used the kaggle.json cell above.
# import subprocess
# subprocess.run([
#     'python', 'scripts/colab_setup.py',
#     '--repo_dir', str(REPO_DIR),
#     '--download_kaggle',
#     '--require_official_splits',
# ], check=True)


## Prepare Flickr8k

This calls the shared Colab setup helper, then enforces the standard paper split counts: train `6000`, validation `1000`, test `1000`. If the source has captions but no split files, the helper downloads the official Flickr8k text archive and copies the standard split files; the notebook fails instead of generating random splits.


In [ ]:
import json
import subprocess
import sys
import config
from pathlib import Path

sys.path.insert(0, str(REPO_DIR))
os.chdir(REPO_DIR)

data_root = REPO_DIR / 'data' / 'flickr8k'
source_dir = Path(FLICKR8K_SOURCE_DIR).expanduser().resolve() if FLICKR8K_SOURCE_DIR else data_root.resolve()

cmd = [
    'python', 'scripts/colab_setup.py',
    '--repo_dir', str(REPO_DIR),
    '--data_root', 'data/flickr8k',
    '--source_dir', str(source_dir),
    '--require_official_splits',
]
subprocess.run(cmd, check=True)

from utils import Vocabulary, validate_dataset_layout
validate_dataset_layout('data/flickr8k', strict_split_counts=True)

vocab_path = data_root / 'vocab.json'
if vocab_path.exists():
    vocab = Vocabulary.load(str(vocab_path))
    if REBUILD_STALE_VOCAB and len(vocab) != config.VOCAB_SIZE:
        print(f'Removing stale vocab ({len(vocab)} tokens); train.py will rebuild {config.VOCAB_SIZE}-token vocab.')
        vocab_path.unlink()
    else:
        assert len(vocab) == config.VOCAB_SIZE, f'Expected vocab size {config.VOCAB_SIZE}, got {len(vocab)}'

print('Dataset and vocab preflight complete.')


## Validate Code

In [ ]:
!pytest -q


## Train

Runs `train.py` with `config.py` exactly as verified above. Training performs validation every epoch and writes `checkpoints/best.pt` using validation BLEU-4.


In [ ]:
!python train.py


## Validate Best Checkpoint and Evaluate Test Set

The training loop already validates each epoch. These two commands re-evaluate the saved best checkpoint on validation and test splits with the same greedy decoding protocol used for the paper baseline table.


In [ ]:
!python evaluate.py   --checkpoint checkpoints/best.pt   --data_root data/flickr8k   --vocab data/flickr8k/vocab.json   --split val   --beam_width 1   --batch_size 64   --results_out results/val_bleu.json

!python evaluate.py   --checkpoint checkpoints/best.pt   --data_root data/flickr8k   --vocab data/flickr8k/vocab.json   --split test   --beam_width 1   --batch_size 64   --results_out results/test_bleu.json


In [ ]:
import json
for result_file in ['results/val_bleu.json', 'results/test_bleu.json']:
    with open(result_file) as f:
        result = json.load(f)
    scores = result['scores']
    print(result_file)
    print({k: round(v * 100, 2) for k, v in scores.items() if k.startswith('bleu')})


## Visualize Attention

In [ ]:
!python visualize.py   --checkpoint checkpoints/best.pt   --vocab data/flickr8k/vocab.json   --data_root data/flickr8k   --split test   --num_examples 6   --output_dir results/attention_examples   --overlay_style paper   --dpi 200


In [ ]:
from IPython.display import Image, display
for path in sorted(Path('results/attention_examples').glob('*.png'))[:6]:
    print(path)
    display(Image(filename=str(path)))
